In [1]:
import pyspiel 
import numpy as np
from open_spiel.python.algorithms import cfr 
from open_spiel.python import rl_environment
from open_spiel.python import rl_tools 
from open_spiel.python.algorithms import random_agent
from open_spiel.python import policy as policy_lib
from open_spiel.python.algorithms import exploitability
from open_spiel.python.algorithms import evaluate_bots
from open_spiel.python.algorithms import cfr_br
import random
import matplotlib.pyplot as plt
import seaborn as sns

In [16]:
def value_func(val, alpha, beta, lambdaa):
    if val >= 0: 
        return val ** alpha 
    return -lambdaa * ((-val) ** beta)

In [17]:
def new_reach(so_far, player, action_prob):
  """Returns new reach probabilities."""
  new = np.array(so_far)
  new[player] *= action_prob
  return new

def calc_cfr(state, reach):
  """Updates regrets; returns utility for all players."""
  if state.is_terminal():
    utilities = np.array([value_func(u, alpha, beta, lambdaa) for u in state.returns()])
    return utilities
  elif state.is_chance_node():
    return sum(prob * calc_cfr(state.child(action), new_reach(reach, -1, prob))
               for action, prob in state.chance_outcomes())
  else:
    # We are at a player decision point.
    player = state.current_player()
    index = policy.state_index(state)
    
    # Compute utilities after each action, updating regrets deeper in the tree.
    utility = np.zeros((game.num_distinct_actions(), game.num_players()))
    for action in state.legal_actions():
      prob = curr_policy[index][action]
      utility[action] = calc_cfr(state.child(action), new_reach(reach, player, prob))

    # Compute regrets at this state.
    cfr_prob = np.prod(reach[:player]) * np.prod(reach[player+1:])
    value = np.einsum('ap,a->p', utility, curr_policy[index])
    for action in state.legal_actions():
      regrets[index][action] += cfr_prob * (utility[action][player] - value[player])

    # Return the value of this state for all players.
    return value

In [21]:
lambdaa, alpha, beta = 2.25, 0.88, 0.88 
game = pyspiel.load_game('leduc_poker')
policy = policy_lib.TabularPolicy(game)
initial_state = game.new_initial_state()
curr_policy = policy.action_probability_array.copy()
regrets = np.zeros_like(policy.action_probability_array)
eval_steps = []
eval_nash_conv = []
for step in range(2049):
  # Compute regrets
  calc_cfr(initial_state, np.ones(1 + game.num_players()))

  # Find the new regret-matching policy
  floored_regrets = np.maximum(regrets, 1e-16)
  sum_floored_regrets = np.sum(floored_regrets, axis=1, keepdims=True)
  curr_policy = floored_regrets / sum_floored_regrets

  # Update the average policy
  lr = 1 / (1 + step)
  policy.action_probability_array *= (1 - lr)
  policy.action_probability_array += curr_policy * lr

  # Evaluate the average policy
  if step & (step-1) == 0:
    nc = exploitability.nash_conv(game, policy)
    eval_steps.append(step)
    eval_nash_conv.append(nc)
    print(f'Nash conv after step {step} is {nc}')

Nash conv after step 0 is 4.695774156365054
Nash conv after step 1 is 4.722849237419357
Nash conv after step 2 is 3.8375145139237774
Nash conv after step 4 is 2.439739546241128
Nash conv after step 8 is 1.7976525049737324
Nash conv after step 16 is 1.6339121081389305
Nash conv after step 32 is 1.7823598686708046
Nash conv after step 64 is 1.7382227970019524
Nash conv after step 128 is 1.4382060744633478
Nash conv after step 256 is 1.6002020568181825
Nash conv after step 512 is 1.6827414124938238
Nash conv after step 1024 is 1.7063398175399633
Nash conv after step 2048 is 1.6566796602193543
